In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Bronze Layer Ingestion
# MAGIC Reads landing_orders → writes bronze_orders with disk partitioning + CDF.

from pyspark.sql.functions import *

dbutils.widgets.text("catalog", "delta_catalog")
dbutils.widgets.text("schema",  "delta_demo")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

# COMMAND ----------
# MAGIC %md ## Read landing data

landing_df = spark.table(f"{CATALOG}.{SCHEMA}.landing_orders")
print("Landing rows:", landing_df.count())

# COMMAND ----------
# MAGIC %md ## Repartition before write — same category groups together

bronze_to_write = landing_df.repartition(6, "category")

print("In-memory partitions after repartition(6, 'category'):",
      bronze_to_write.select(spark_partition_id()).distinct().count())

# COMMAND ----------
# MAGIC %md ## Create Bronze table with partitioning + CDF

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.bronze_orders (
        order_id    BIGINT,
        customer_id INT,
        product_id  INT,
        category    STRING,
        amount      INT,
        order_date  DATE,
        status      STRING,
        year        INT,
        month       INT
    )
    USING DELTA
    PARTITIONED BY (year, month)
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

bronze_to_write.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_orders")

# COMMAND ----------
# MAGIC %md ## Verify partition pruning

pruned = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders") \
              .filter("year = 2025 AND month = 1")
pruned.explain()

print("Jan 2025 orders:", pruned.count())

# dbutils.notebook.exit(f"01_bronze: SUCCESS - {landing_df.count()} rows")